# Standalone Prophet Notebook

This version is rebuilt to be lighter on RAM for all `1,000` tickers in Colab.

What changed:
- the notebook no longer loads the full SQLite table into one giant DataFrame
- all train/test splitting and feature engineering are done ticker-by-ticker
- cross-validation and final fitting operate on per-ticker slices only

Expected training time on the current dataset:
- Full dataset, 3 CV folds + final per-ticker fit: about 1-3.5 hours
- Prophet is CPU-bound. Colab GPU does not help here.

The saved `.pkl` artifact is still compatible with the pipeline's `prophet` loader.


In [ ]:
!pip -q install pandas numpy scikit-learn prophet joblib pyarrow


In [ ]:
from pathlib import Path

DB_PATH = Path('/content/stock_prices_20y.db')
OUTPUT_DIR = Path('/content/model_outputs')
MODELS_DIR = OUTPUT_DIR / 'models'
RESULTS_DIR = OUTPUT_DIR / 'results'
CONFIGS_DIR = OUTPUT_DIR / 'configs'
CACHE_DIR = OUTPUT_DIR / 'prophet_cache'

for path in [MODELS_DIR, RESULTS_DIR, CONFIGS_DIR, CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not DB_PATH.exists():
    raise FileNotFoundError(f'Missing SQLite database: {DB_PATH}')

print('DB_PATH    =', DB_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('CACHE_DIR  =', CACHE_DIR)


In [ ]:
import json
import os
import pickle
import sqlite3
import time
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from prophet import Prophet
from prophet.serialize import model_to_json
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
RUN_TAG = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
REGRESSOR_COLS = ['log_volume', 'log_ret_1', 'hl_range', 'rsi_14']
N_WORKERS = min(4, os.cpu_count() or 1)

with sqlite3.connect(DB_PATH) as conn:
    row_count = conn.execute('SELECT COUNT(*) FROM prices').fetchone()[0]
    ticker_count = conn.execute('SELECT COUNT(DISTINCT ticker) FROM prices').fetchone()[0]
    min_date, max_date = conn.execute('SELECT MIN(date), MAX(date) FROM prices').fetchone()
print(f'Rows: {row_count:,} | Tickers: {ticker_count:,} | Date range: {min_date} -> {max_date}')

@dataclass
class Config:
    db_path: str = str(DB_PATH)
    table_name: str = 'prices'
    n_splits: int = 3
    test_size: float = 0.2
    target_horizon: int = 1
    random_state: int = 42
    use_log_target: bool = False
    changepoint_prior_scale: float = 0.2
    seasonality_prior_scale: float = 15.0
    yearly_seasonality: bool = True
    weekly_seasonality: bool = True
    daily_seasonality: bool = False
    seasonality_mode: str = 'multiplicative'
    max_tickers: int = 0
    cache_path: str = str(CACHE_DIR / 'ticker_splits.pkl')
    model_output_path: str = str(MODELS_DIR / f'{RUN_TAG}-prophet.pkl')
    config_output_path: str = str(CONFIGS_DIR / f'{RUN_TAG}-prophet_config.json')

cfg = Config()
print(cfg)
print('Prophet workers:', N_WORKERS)


def get_tickers(conn: sqlite3.Connection) -> list[str]:
    return [r[0] for r in conn.execute(f'SELECT DISTINCT ticker FROM {cfg.table_name} ORDER BY ticker')]


def load_ticker_prices(conn: sqlite3.Connection, ticker: str) -> pd.DataFrame:
    df = pd.read_sql_query(
        f'''SELECT ticker, date, open, high, low, close, adj_close, volume
            FROM {cfg.table_name}
            WHERE ticker = ?
            ORDER BY date''',
        conn,
        params=[ticker],
    )
    df['date'] = pd.to_datetime(df['date'])
    for c in ['open', 'high', 'low', 'close', 'adj_close']:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')
    df['volume'] = pd.to_numeric(df['volume'], errors='coerce').fillna(0).astype('float32')
    return df


def _compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0).ewm(alpha=1.0 / period, min_periods=period).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1.0 / period, min_periods=period).mean()
    rs = gain / (loss + 1e-8)
    return 100.0 - 100.0 / (1.0 + rs)


def add_features_single_ticker(df: pd.DataFrame) -> pd.DataFrame:
    g = df.sort_values('date').copy()
    close = g['close'].astype('float64')
    volume = g['volume'].astype('float64').clip(lower=1.0)
    g['log_ret_1'] = np.log(close / close.shift(1))
    g['log_volume'] = np.log1p(volume)
    g['hl_range'] = (g['high'].astype('float64') - g['low'].astype('float64')) / close
    g['rsi_14'] = _compute_rsi(close, 14) / 100.0
    return g.replace([np.inf, -np.inf], np.nan)


def split_train_test_single_ticker(df: pd.DataFrame, test_size: float, embargo: int):
    g = df.sort_values('date').reset_index(drop=True)
    n = len(g)
    split_idx = int(n * (1 - test_size))
    train_end = split_idx - embargo
    if split_idx <= 0 or split_idx >= n or train_end <= 0:
        return None, None
    return g.iloc[:train_end].copy(), g.iloc[split_idx:].copy()


def build_expanding_index_folds(n_rows: int, n_splits: int, embargo: int):
    folds = []
    if n_rows < (n_splits + 1) * max(embargo, 1) + 10:
        return folds
    boundaries = np.linspace(0, n_rows, n_splits + 2, dtype=int)
    for fold_num in range(n_splits):
        va_start = boundaries[fold_num + 1]
        va_end = boundaries[fold_num + 2]
        purged_train_end = va_start - embargo
        if purged_train_end <= 0 or va_end <= va_start:
            continue
        folds.append((slice(0, purged_train_end), slice(va_start, va_end)))
    return folds


def regression_metrics_with_direction(y_true: np.ndarray, y_pred: np.ndarray, prev_values: np.ndarray) -> dict:
    mask = np.isfinite(y_true) & np.isfinite(y_pred) & np.isfinite(prev_values)
    y_true, y_pred, prev_values = y_true[mask], y_pred[mask], prev_values[mask]
    if len(y_true) < 2:
        return {'rmse': np.nan, 'mae': np.nan, 'r2': np.nan, 'directional_accuracy': np.nan}
    da = float(np.mean(np.sign(y_pred - prev_values) == np.sign(y_true - prev_values)))
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'directional_accuracy': da,
    }


def fit_prophet_for_ticker(train_ticker_df: pd.DataFrame, cfg: Config) -> Prophet:
    close = train_ticker_df['close'].astype('float64').clip(lower=1e-6)
    y_vals = np.log(close) if cfg.use_log_target else close.values
    prophet_df = pd.DataFrame({'ds': train_ticker_df['date'].values, 'y': y_vals})
    for col in REGRESSOR_COLS:
        vals = train_ticker_df[col].values.astype('float64') if col in train_ticker_df.columns else np.zeros(len(train_ticker_df))
        prophet_df[col] = np.where(np.isfinite(vals), vals, 0.0)
    prophet_df = prophet_df.dropna(subset=['ds', 'y']).copy()
    if len(prophet_df) < 30:
        raise ValueError('Not enough rows to fit Prophet.')
    model = Prophet(
        changepoint_prior_scale=cfg.changepoint_prior_scale,
        seasonality_prior_scale=cfg.seasonality_prior_scale,
        yearly_seasonality=cfg.yearly_seasonality,
        weekly_seasonality=cfg.weekly_seasonality,
        daily_seasonality=cfg.daily_seasonality,
        seasonality_mode=cfg.seasonality_mode,
    )
    for col in REGRESSOR_COLS:
        model.add_regressor(col, standardize=True)
    model.fit(prophet_df)
    return model


def predict_prophet_for_ticker(model: Prophet, eval_df: pd.DataFrame, cfg: Config) -> np.ndarray:
    future_df = pd.DataFrame({'ds': eval_df['date'].values})
    for col in REGRESSOR_COLS:
        vals = eval_df[col].values.astype('float64') if col in eval_df.columns else np.zeros(len(eval_df))
        future_df[col] = np.where(np.isfinite(vals), vals, 0.0)
    forecast = model.predict(future_df)
    yhat = forecast['yhat'].values.astype(np.float64)
    if cfg.use_log_target:
        yhat = np.clip(yhat, -100.0, 20.0)
        return np.exp(yhat).astype(np.float32)
    return yhat.astype(np.float32)


def run_prophet_for_ticker(train_ticker_df, eval_ticker_df, cfg, store_model=False):
    ticker = str(train_ticker_df['ticker'].iloc[0])
    try:
        model = fit_prophet_for_ticker(train_ticker_df, cfg)
        pred_close = predict_prophet_for_ticker(model, eval_ticker_df, cfg)
        actual_close = eval_ticker_df['close'].to_numpy(dtype=np.float64)
        last_train_close = float(train_ticker_df['close'].iloc[-1])
        prev_close = np.concatenate([[last_train_close], actual_close[:-1]])
        result = {'ticker': ticker, 'pred_close': pred_close, 'actual_close': actual_close, 'prev_close': prev_close}
        if store_model:
            result['model_json'] = model_to_json(model)
            result['dates'] = eval_ticker_df['date'].to_numpy()
        return result
    except Exception as exc:
        return {'ticker': ticker, 'error': str(exc)}


def build_ticker_splits(cfg: Config):
    cache_path = Path(cfg.cache_path)
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            cached = pickle.load(f)
        print('Loaded cached ticker splits from', cache_path)
        return cached
    with sqlite3.connect(cfg.db_path) as conn:
        tickers = get_tickers(conn)
        if cfg.max_tickers > 0 and len(tickers) > cfg.max_tickers:
            rng = np.random.default_rng(cfg.random_state)
            tickers = rng.choice(tickers, cfg.max_tickers, replace=False).tolist()
        data = {}
        for idx, ticker in enumerate(tickers, start=1):
            raw = load_ticker_prices(conn, ticker)
            feat = add_features_single_ticker(raw)
            train_df, test_df = split_train_test_single_ticker(feat, cfg.test_size, cfg.target_horizon)
            if train_df is None or len(train_df) < 30 or len(test_df) < 2:
                continue
            fold_pairs = []
            for tr_slice, va_slice in build_expanding_index_folds(len(train_df), cfg.n_splits, cfg.target_horizon):
                tr_t = train_df.iloc[tr_slice].copy()
                va_t = train_df.iloc[va_slice].copy()
                if len(tr_t) >= 30 and len(va_t) >= 2:
                    fold_pairs.append((tr_t, va_t))
            data[ticker] = {'train': train_df, 'test': test_df, 'folds': fold_pairs}
            if idx % 50 == 0:
                print(f'Prepared {idx}/{len(tickers)} tickers | usable={len(data)}')
    with open(cache_path, 'wb') as f:
        pickle.dump(data, f)
    print('Saved ticker splits cache to', cache_path)
    return data


In [ ]:
start_time = time.perf_counter()
PROGRESS_CHUNK = 25

def run_tasks_with_progress(tasks, store_model=False, label='Progress'):
    outputs = []
    total = len(tasks)
    if total == 0:
        return outputs
    for start in range(0, total, PROGRESS_CHUNK):
        batch = tasks[start:start + PROGRESS_CHUNK]
        batch_outputs = Parallel(n_jobs=N_WORKERS, backend='loky')(
            delayed(run_prophet_for_ticker)(tr_t, ev_t, cfg, store_model)
            for tr_t, ev_t in batch
        )
        outputs.extend(batch_outputs)
        done = min(start + len(batch), total)
        print(f'{label}: {done}/{total} tickers processed')
    return outputs

ticker_data = build_ticker_splits(cfg)
print(f'Usable tickers: {len(ticker_data):,}')

cv_results = []
for fold_i in range(cfg.n_splits):
    print(f'\n=== Fold {fold_i + 1}/{cfg.n_splits} ===')
    tasks = []
    for ticker, item in ticker_data.items():
        if fold_i < len(item['folds']):
            tr_t, va_t = item['folds'][fold_i]
            tasks.append((tr_t, va_t))
    print(f'Fold {fold_i + 1}: {len(tasks):,} ticker jobs queued')
    fold_outputs = run_tasks_with_progress(tasks, store_model=False, label=f'Fold {fold_i + 1}')
    fold_preds, fold_trues, fold_prevs = [], [], []
    for item in fold_outputs:
        if 'error' in item:
            continue
        fold_preds.extend(item['pred_close'].tolist())
        fold_trues.extend(item['actual_close'].tolist())
        fold_prevs.extend(item['prev_close'].tolist())
    if fold_trues:
        metrics = regression_metrics_with_direction(np.asarray(fold_trues), np.asarray(fold_preds), np.asarray(fold_prevs))
        cv_results.append(metrics)
        print(metrics)
    else:
        print('No valid fold predictions')

cv_df = pd.DataFrame(cv_results)
if not cv_df.empty:
    print('\nCV mean metrics')
    print(cv_df.mean(numeric_only=True).to_string())
else:
    print('\nNo CV metrics were produced.')

print('\nTraining final per-ticker Prophet models...')
final_tasks = [(item['train'], item['test']) for item in ticker_data.values()]
print(f'Final fit: {len(final_tasks):,} ticker jobs queued')
final_outputs = run_tasks_with_progress(final_tasks, store_model=True, label='Final fit')

all_preds, all_trues, all_prevs, all_meta = [], [], [], []
models_by_ticker = {}
for item in final_outputs:
    if 'error' in item:
        continue
    ticker = item['ticker']
    models_by_ticker[ticker] = item['model_json']
    dates = item['dates']
    actual_close = item['actual_close']
    pred_close = item['pred_close']
    prev_close = item['prev_close']
    for idx in range(len(actual_close)):
        all_preds.append(float(pred_close[idx]))
        all_trues.append(float(actual_close[idx]))
        all_prevs.append(float(prev_close[idx]))
        all_meta.append((ticker, dates[idx], float(actual_close[idx])))

if not all_trues:
    raise RuntimeError('No Prophet models were trained successfully.')

y_true = np.asarray(all_trues, dtype=np.float64)
y_pred = np.asarray(all_preds, dtype=np.float64)
y_prev = np.asarray(all_prevs, dtype=np.float64)
test_metrics = regression_metrics_with_direction(y_true, y_pred, y_prev)
print('Held-out test metrics:', test_metrics)

payload = {'models_by_ticker': models_by_ticker, 'regressor_cols': REGRESSOR_COLS, 'config': asdict(cfg), 'cv_results': cv_results, 'test_metrics': test_metrics, 'trained_tickers': sorted(models_by_ticker)}
with open(cfg.model_output_path, 'wb') as f:
    pickle.dump(payload, f)
with open(cfg.config_output_path, 'w') as f:
    json.dump(asdict(cfg), f, indent=2)

pred_path = RESULTS_DIR / f'{RUN_TAG}-prophet_predictions.csv'
meta_df = pd.DataFrame(all_meta, columns=['ticker', 'date', 'close'])
meta_df['y_true'] = y_true
meta_df['y_pred'] = y_pred
meta_df['y_prev'] = y_prev
meta_df.to_csv(pred_path, index=False)

elapsed_min = (time.perf_counter() - start_time) / 60
print(f'\nSaved model      : {cfg.model_output_path}')
print(f'Saved config     : {cfg.config_output_path}')
print(f'Saved predictions: {pred_path}')
print(f'Trained tickers  : {len(models_by_ticker):,}')
print(f'Elapsed minutes  : {elapsed_min:.2f}')
